# 圖像卷積
:label:`sec_conv_layer`

上節我們解析了卷積層的原理，現在我們看看它的實際應用。由於卷積神經網路的設計是用於探索圖像數據，本節我們將以圖像為例。

## 互相關運算

嚴格來說，卷積層是個錯誤的叫法，因為它所表達的運算其實是*互相關運算*（cross-correlation），而不是卷積運算。
根據 :numref:`sec_why-conv`中的描述，在卷積層中，輸入張量和核張量通過(**互相關運算**)產生輸出張量。

首先，我們暫時忽略通道（第三維）這一情況，看看如何處理二維圖像數據和隱藏表示。在 :numref:`fig_correlation`中，輸入是高度為$3$、寬度為$3$的二維張量（即形狀為$3 \times 3$）。卷積核的高度和寬度都是$2$，而卷積核窗口（或卷積窗口）的形狀由內核的高度和寬度決定（即$2 \times 2$）。

![二維互相關運算。陰影部分是第一個輸出元素，以及用於計算輸出的輸入張量元素和核張量元素：$0\times0+1\times1+3\times2+4\times3=19$.](../img/correlation.svg)
:label:`fig_correlation`

在二維互相關運算中，卷積窗口從輸入張量的左上角開始，從左到右、從上到下滑動。
當卷積窗口滑動到新一個位置時，包含在該窗口中的部分張量與卷積核張量進行按元素相乘，得到的張量再求和得到一個單一的標量值，由此我們得出了這一位置的輸出張量值。
在如上例子中，輸出張量的四個元素由二維互相關運算得到，這個輸出高度為$2$、寬度為$2$，如下所示：

$$
0\times0+1\times1+3\times2+4\times3=19,\\
1\times0+2\times1+4\times2+5\times3=25,\\
3\times0+4\times1+6\times2+7\times3=37,\\
4\times0+5\times1+7\times2+8\times3=43.
$$

注意，輸出大小略小於輸入大小。這是因為卷積核的寬度和高度大於1，
而卷積核只與圖像中每個大小完全適合的位置進行互相關運算。
所以，輸出大小等於輸入大小$n_h \times n_w$減去卷積核大小$k_h \times k_w$，即：

$$(n_h-k_h+1) \times (n_w-k_w+1).$$

這是因為我們需要足夠的空間在圖像上“移動”卷積核。稍後，我們將看到如何通過在圖像邊界周圍填充零來保證有足夠的空間移動卷積核，從而保持輸出大小不變。
接下來，我們在`corr2d`函數中實現如上過程，該函數接受輸入張量`X`和卷積核張量`K`，並返回輸出張量`Y`。


In [1]:
import torch
from torch import nn

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
def corr2d(X, K):  #@save
    """計算二維互相關運算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

通過 :numref:`fig_correlation`的輸入張量`X`和卷積核張量`K`，我們來[**驗證上述二維互相關運算的輸出**]。


In [3]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

## 卷積層

卷積層對輸入和卷積核權重進行互相關運算，並在添加標量偏置之後產生輸出。
所以，卷積層中的兩個被訓練的參數是卷積核權重和標量偏置。
就像我們之前隨機初始化全連接層一樣，在訓練基於卷積層的模型時，我們也隨機初始化卷積核權重。

基於上面定義的`corr2d`函數[**實現二維卷積層**]。在`__init__`構造函數中，將`weight`和`bias`聲明為兩個模型參數。前向傳播函數調用`corr2d`函數並添加偏置。


In [4]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

高度和寬度分別為$h$和$w$的卷積核可以被稱為$h \times w$卷積或$h \times w$卷積核。
我們也將帶有$h \times w$卷積核的卷積層稱為$h \times w$卷積層。

## 圖像中目標的邊緣檢測

如下是[**卷積層的一個簡單應用：**]通過找到像素變化的位置，來(**檢測圖像中不同顏色的邊緣**)。
首先，我們構造一個$6\times 8$像素的黑白圖像。中間四列為黑色（$0$），其餘像素為白色（$1$）。


In [5]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

接下來，我們構造一個高度為$1$、寬度為$2$的卷積核`K`。當進行互相關運算時，如果水平相鄰的兩元素相同，則輸出為零，否則輸出為非零。


In [6]:
K = torch.tensor([[1.0, -1.0]])

現在，我們對參數`X`（輸入）和`K`（卷積核）執行互相關運算。
如下所示，[**輸出`Y`中的1代表從白色到黑色的邊緣，-1代表從黑色到白色的邊緣**]，其他情況的輸出為$0$。


In [7]:
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

現在我們將輸入的二維圖像轉置，再進行如上的互相關運算。
其輸出如下，之前檢測到的垂直邊緣消失了。
不出所料，這個[**卷積核`K`只可以檢測垂直邊緣**]，無法檢測水平邊緣。


In [8]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

## 學習卷積核

如果我們只需尋找黑白邊緣，那麼以上`[1, -1]`的邊緣檢測器足以。然而，當有了更複雜數值的卷積核，或者連續的卷積層時，我們不可能手動設計濾波器。那麼我們是否可以[**學習由`X`生成`Y`的卷積核**]呢？

現在我們看看是否可以通過僅查看“輸入-輸出”對來學習由`X`生成`Y`的卷積核。
我們先構造一個卷積層，並將其卷積核初始化為隨機張量。接下來，在每次迭代中，我們比較`Y`與卷積層輸出的平方誤差，然後計算梯度來更新卷積核。為了簡單起見，我們在此使用內置的二維卷積層，並忽略偏置。


In [9]:
# 構造一個二維卷積層，它具有1個輸出通道和形狀為（1，2）的卷積核
conv2d = nn.Conv2d(1,1, kernel_size=(1, 2), bias=False)

# 這個二維卷積層使用四維輸入和輸出格式（批量大小、通道、高度、寬度），
# 其中批量大小和通道數都為1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # 學習率

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # 迭代卷積核
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 10.390
epoch 4, loss 1.873
epoch 6, loss 0.367
epoch 8, loss 0.083
epoch 10, loss 0.023


[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


在$10$次迭代之後，誤差已經降到足夠低。現在我們來看看我們[**所學的卷積核的權重張量**]。


In [10]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 0.9710, -0.9964]])

細心的讀者一定會發現，我們學習到的卷積核權重非常接近我們之前定義的卷積核`K`。

## 互相關和卷積

回想一下我們在 :numref:`sec_why-conv`中觀察到的互相關和卷積運算之間的對應關係。
為了得到正式的*卷積*運算輸出，我們需要執行 :eqref:`eq_2d-conv-discrete`中定義的嚴格卷積運算，而不是互相關運算。
幸運的是，它們差別不大，我們只需水平和垂直翻轉二維卷積核張量，然後對輸入張量執行*互相關*運算。

值得注意的是，由於卷積核是從數據中學習到的，因此無論這些層執行嚴格的卷積運算還是互相關運算，卷積層的輸出都不會受到影響。
為了說明這一點，假設卷積層執行*互相關*運算並學習 :numref:`fig_correlation`中的卷積核，該卷積核在這裡由矩陣$\mathbf{K}$表示。
假設其他條件不變，當這個層執行嚴格的*卷積*時，學習到的卷積核$\mathbf{K}'$在水平和垂直翻轉之後將與$\mathbf{K}$相同。
也就是說，當卷積層對 :numref:`fig_correlation`中的輸入和$\mathbf{K}'$執行嚴格的*卷積*運算時，將得到與互相關運算 :numref:`fig_correlation`中相同的輸出。

為了與深度學習文獻中的標準術語保持一致，我們將繼續把“互相關運算”稱為卷積運算，儘管嚴格地說，它們略有不同。
此外，對於卷積核張量上的權重，我們稱其為*元素*。

## 特徵映射和感受野

如在 :numref:`subsec_why-conv-channels`中所述， :numref:`fig_correlation`中输出的卷积层有时被称为*特徵映射*（feature map），因为它可以被視為一個輸入映射到下一層空間維度的轉換器。
在卷積神經網路中，對於某層的任意元素$x$，其*感受野*（receptive field）是指在前向傳播期間可能影響$x$計算的所有元素（來自所有先前層）。

請注意，感受野可能大於輸入的實際大小。讓我們用 :numref:`fig_correlation`為例來解釋感受野：
給定$2 \times 2$卷積核，陰影輸出元素值$19$的感受野是輸入陰影部分的四個元素。
假設之前輸出為$\mathbf{Y}$，其大小為$2 \times 2$，現在我們在其後附加一個卷積層，該卷積層以$\mathbf{Y}$為輸入，輸出單個元素$z$。
在這種情況下，$\mathbf{Y}$上的$z$的感受野包括$\mathbf{Y}$的所有四個元素，而輸入的感受野包括最初所有九個輸入元素。
因此，當一個特徵圖中的任意元素需要檢測更廣區域的輸入特徵時，我們可以構建一個更深的網路。

## 小節總結

* 二維卷積層的核心計算是二維互相關運算。最簡單的形式是，對二維輸入數據和卷積核執行互相關操作，然後添加一個偏置。
* 我們可以設計一個卷積核來檢測圖像的邊緣。
* 我們可以從數據中學習卷積核的參數。
* 學習卷積核時，無論用嚴格卷積運算或互相關運算，卷積層的輸出不會受太大影響。
* 當需要檢測輸入特徵中更廣區域時，我們可以構建一個更深的卷積網路。

## 練習

1. 構建一個具有對角線邊緣的圖像`X`。
    1. 如果將本節中舉例的卷積核`K`應用於`X`，會發生什麼情況？
    1. 如果轉置`X`會發生什麼？
    1. 如果轉置`K`會發生什麼？
1. 在我們創建的`Conv2D`自動求導時，有什麼錯誤消息？
1. 如何通過改變輸入張量和卷積核張量，將互相關運算表示為矩陣乘法？
1. 手工設計一些卷積核。
    1. 二階導數的核的形式是什麼？
    1. 積分的核的形式是什么？
    1. 得到$d$次導數的最小核的大小是多少？


[Discussions](https://discuss.d2l.ai/t/1848)


練習一：

1. 構建一個具有對角線邊緣的圖像`X`。
    1. 如果將本節中舉例的卷積核`K`應用於`X`，會發生什麼情況？
    1. 如果轉置`X`會發生什麼？
    1. 如果轉置`K`會發生什麼？

我的回答：

讓我們逐步解答這些問題：

```python
import torch

# 1. 構建對角線邊緣的圖像
X = torch.zeros((6, 6))
for i in range(6):
    X[i, i] = 1  # 主對角線設為1
print("原始圖像 X:")
print(X)

# 定義卷積核 K (用於檢測垂直邊緣的核)
K = torch.tensor([[1.0, -1.0]])

# 定義互相關運算函數
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

# 1.1 應用卷積核K
Y1 = corr2d(X, K)
print("\n應用K後的結果:")
print(Y1)

# 1.2 轉置X後應用K
X_t = X.t()
print("\n轉置後的X:")
print(X_t)
Y2 = corr2d(X_t, K)
print("\n對轉置後的X應用K的結果:")
print(Y2)

# 1.3 轉置K後應用於原始X
K_t = K.t()
print("\n轉置後的K:")
print(K_t)
Y3 = corr2d(X, K_t)
print("\n使用轉置後的K的結果:")
print(Y3)
```

分析結果：

1. 原始卷積核K應用於X：
- K是檢測垂直邊緣的核
- 對角線包含了45度的邊緣
- 輸出會顯示對角線附近的變化

2. 轉置X後應用K：
- 轉置X會將對角線保持不變
- 但是相對於垂直方向的檢測效果不變
- 結果應該與原始結果相似

3. 轉置K後應用：
- K_t變成垂直的核 [[1], [-1]]
- 現在檢測水平邊緣
- 結果會顯示水平方向的變化

主要觀察：
1. 原始K只能檢測垂直邊緣
2. 對角線邊緣會產生較弱的響應
3. 轉置操作改變了邊緣檢測的方向

這個例子很好地說明了卷積核的方向性和它們對不同類型邊緣的響應特性。


練習二：

2. 在我們創建的`Conv2D`自動求導時，有什麼錯誤消息？

我的回答：



在我們創建的`Conv2D`類中，主要問題出在`corr2d`函數上。讓我們分析並修復這個問題：

1. 原始問題：
````python
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias
````

主要錯誤：
- `corr2d`函數使用了基本的Python循環
- 不支持自動求導
- 不支持批量處理

2. 修復版本：
````python
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # 使用F.conv2d替代corr2d
        # 需要調整輸入維度
        x = x.unsqueeze(0).unsqueeze(0) if x.dim() == 2 else x
        weight = self.weight.unsqueeze(0).unsqueeze(0)
        
        # 執行卷積
        out = F.conv2d(x, weight, self.bias)
        return out.squeeze() if x.dim() == 2 else out

# 測試代碼
def test_conv2d():
    # 創建卷積層
    conv2d = Conv2D((2, 2))
    
    # 創建輸入
    X = torch.randn(3, 3, requires_grad=True)
    
    # 前向傳播
    Y = conv2d(X)
    
    # 反向傳播
    Y.sum().backward()
    
    print("輸入梯度:", X.grad is not None)
    print("權重梯度:", conv2d.weight.grad is not None)
    print("偏置梯度:", conv2d.bias.grad is not None)
    
    return "測試成功！"
````

3. 完整的可求導版本：
````python
class Conv2D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
            
        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, *kernel_size) / 
            (in_channels * kernel_size[0] * kernel_size[1])**0.5
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))
        
    def forward(self, x):
        # 處理輸入維度
        if x.dim() == 2:
            x = x.unsqueeze(0).unsqueeze(0)
        elif x.dim() == 3:
            x = x.unsqueeze(1)
            
        # 執行卷積
        out = F.conv2d(x, self.weight, self.bias)
        
        # 恢復原始維度
        if x.dim() <= 3:
            out = out.squeeze()
            
        return out
    
    def extra_repr(self):
        return f'weight_shape={tuple(self.weight.shape)}'

# 更完整的測試
def comprehensive_test():
    # 測試不同輸入維度
    test_cases = [
        torch.randn(3, 3),  # 2D輸入
        torch.randn(1, 3, 3),  # 3D輸入
        torch.randn(2, 1, 3, 3)  # 4D輸入
    ]
    
    conv2d = Conv2D(1, 1, 2)
    
    for i, x in enumerate(test_cases):
        print(f"\n測試案例 {i+1}:")
        print(f"輸入形狀: {x.shape}")
        x.requires_grad = True
        
        try:
            y = conv2d(x)
            print(f"輸出形狀: {y.shape}")
            
            # 測試反向傳播
            y.sum().backward()
            print("反向傳播成功")
            print(f"輸入梯度形狀: {x.grad.shape}")
            
        except Exception as e:
            print(f"錯誤: {e}")
````

主要改進：
1. 使用`F.conv2d`替代手動實現
2. 正確處理輸入維度
3. 添加適當的初始化
4. 支持批量處理
5. 完整的梯度計算

這樣修改後，Conv2D類可以：
- 正確計算梯度
- 支持不同輸入維度
- 與PyTorch的自動求導系統集成
- 提供更好的數值穩定性


練習三：

3. 如何通過改變輸入張量和卷積核張量，將互相關運算表示為矩陣乘法？

我的回答：



讓我解釋如何將互相關運算轉換為矩陣乘法：

`````python
import torch
import torch.nn.functional as F

def conv_to_matrix(x, k_h, k_w):
    """將輸入張量轉換為可進行矩陣乘法的形式"""
    batch_size, in_channels, height, width = x.shape
    out_height = height - k_h + 1
    out_width = width - k_w + 1
    
    # 創建滑動窗口視圖
    strided = F.unfold(x, (k_h, k_w))
    # 重塑為矩陣形式
    matrix = strided.transpose(1, 2)
    
    return matrix, (out_height, out_width)

def kernel_to_matrix(kernel):
    """將卷積核轉換為矩陣形式"""
    out_channels, in_channels, k_h, k_w = kernel.shape
    # 重塑卷積核為矩陣形式
    return kernel.view(out_channels, -1)

# 示例實現
def conv2d_as_matrix_mult():
    # 創建示例輸入
    batch_size, in_channels = 2, 3
    height, width = 5, 5
    x = torch.randn(batch_size, in_channels, height, width)
    
    # 創建卷積核
    out_channels = 2
    k_h, k_w = 3, 3
    kernel = torch.randn(out_channels, in_channels, k_h, k_w)
    
    # 1. 標準卷積
    conv_result = F.conv2d(x, kernel)
    
    # 2. 矩陣乘法實現
    # 轉換輸入
    x_matrix, (out_h, out_w) = conv_to_matrix(x, k_h, k_w)
    # 轉換卷積核
    k_matrix = kernel_to_matrix(kernel)
    
    # 執行矩陣乘法
    result = torch.matmul(x_matrix, k_matrix.t())
    # 重塑回原始形狀
    matrix_result = result.view(batch_size, out_h, out_w, out_channels)
    matrix_result = matrix_result.permute(0, 3, 1, 2)
    
    # 驗證結果
    print("結果是否相同:", torch.allclose(conv_result, matrix_result))
    
    return conv_result, matrix_result

# 更詳細的實現，包含步長和填充
class MatrixConv2d(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, 
                 stride=1, padding=0):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        # 初始化權重
        self.weight = torch.nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size)
        )
        self.bias = torch.nn.Parameter(torch.zeros(out_channels))
        
    def forward(self, x):
        batch_size = x.size(0)
        
        # 添加填充
        if self.padding > 0:
            x = F.pad(x, (self.padding,) * 4)
        
        # 轉換為矩陣形式
        x_unfolded = F.unfold(
            x, 
            kernel_size=(self.kernel_size, self.kernel_size),
            stride=self.stride
        )
        
        # 重塑卷積核
        weight_matrix = self.weight.view(self.out_channels, -1)
        
        # 矩陣乘法
        out = torch.matmul(weight_matrix, x_unfolded)
        
        # 添加偏置
        out = out + self.bias.view(-1, 1)
        
        # 重塑輸出
        height = (x.size(2) - self.kernel_size + 2*self.padding) // self.stride + 1
        width = (x.size(3) - self.kernel_size + 2*self.padding) // self.stride + 1
        out = out.view(batch_size, self.out_channels, height, width)
        
        return out
`````

轉換步驟說明：

1. 輸入張量轉換：
- 使用滑動窗口提取所有局部區域
- 將每個局部區域展平為向量
- 將所有向量組織成矩陣

2. 卷積核轉換：
- 將卷積核展平為向量
- 組織成適合矩陣乘法的形狀

3. 計算過程：
- 輸入矩陣和卷積核矩陣相乘
- 重塑結果回原始空間維度
- 添加偏置項

優點：
1. 可以利用高效的矩陣乘法實現
2. 便於理解卷積運算的本質
3. 在某些硬件上可能更高效

缺點：
1. 記憶體使用量較大
2. 對大型輸入可能不夠高效
3. 實現複雜度增加

這種轉換幫助我們：
- 理解卷積運算的本質
- 利用高效的矩陣運算庫
- 在某些情況下提高性能


練習四：

4. 手工設計一些卷積核。
    1. 二階導數的核的形式是什麼？
    1. 積分的核的形式是什么？
    1. 得到$d$次導數的最小核的大小是多少？

我的回答：

讓我們逐一分析這些問題：

1. 二階導數的核：
``````python
import torch
import numpy as np
import matplotlib.pyplot as plt

# 一維二階導數核
kernel_2nd_1d = torch.tensor([1.0, -2.0, 1.0])

# 二維二階導數核（拉普拉斯算子）
kernel_2nd_2d = torch.tensor([
    [0.0,  1.0, 0.0],
    [1.0, -4.0, 1.0],
    [0.0,  1.0, 0.0]
])

def visualize_kernel(kernel, title):
    plt.imshow(kernel.numpy(), cmap='coolwarm')
    plt.colorbar()
    plt.title(title)
    plt.show()

# 測試二階導數核
def test_2nd_derivative():
    # 創建測試信號
    x = torch.linspace(-5, 5, 100)
    y = torch.exp(-x**2/2)  # 高斯函數
    
    # 應用一維二階導數
    conv_result = torch.conv1d(
        y.view(1, 1, -1), 
        kernel_2nd_1d.view(1, 1, -1), 
        padding=1
    )
    
    plt.plot(x, y, label='原始函數')
    plt.plot(x, conv_result.squeeze(), label='二階導數')
    plt.legend()
    plt.title('高斯函數的二階導數')
    plt.show()
``````

2. 積分核：
``````python
# 一維積分核（矩形窗）
kernel_integral_1d = torch.ones(5) / 5  # 簡單移動平均

# 二維積分核（高斯平滑）
def gaussian_kernel(size=5, sigma=1.0):
    x = torch.linspace(-size//2, size//2, size)
    x, y = torch.meshgrid(x, x)
    kernel = torch.exp(-(x**2 + y**2)/(2*sigma**2))
    return kernel / kernel.sum()

kernel_integral_2d = gaussian_kernel()

# 測試積分核
def test_integral():
    # 創建帶噪聲的信號
    x = torch.linspace(0, 10, 100)
    y = torch.sin(x) + torch.randn_like(x) * 0.2
    
    # 應用積分（平滑）
    conv_result = torch.conv1d(
        y.view(1, 1, -1),
        kernel_integral_1d.view(1, 1, -1),
        padding=2
    )
    
    plt.plot(x, y, label='原始信號（帶噪聲）')
    plt.plot(x, conv_result.squeeze(), label='平滑後')
    plt.legend()
    plt.title('使用積分核進行平滑')
    plt.show()
``````

3. d次導數的最小核大小：
``````python
def min_kernel_size(d):
    """計算d次導數的最小核大小"""
    return d + 1

def generate_derivative_kernel(d):
    """生成d次導數的核（一維）"""
    if d == 0:
        return torch.tensor([1.0])
    elif d == 1:
        return torch.tensor([-1.0, 1.0])
    else:
        # 使用遞歸方式生成高階導數核
        prev_kernel = generate_derivative_kernel(d-1)
        kernel = torch.conv1d(
            prev_kernel.view(1, 1, -1),
            torch.tensor([-1.0, 1.0]).view(1, 1, -1)
        ).squeeze()
        return kernel

# 測試不同階數的導數核
def test_derivative_kernels():
    for d in range(4):
        kernel = generate_derivative_kernel(d)
        print(f"{d}階導數核（大小={len(kernel)}):")
        print(kernel)
``````

主要結論：

1. 二階導數核：
- 一維：[1, -2, 1]
- 二維：使用拉普拉斯算子
- 可以檢測邊緣和曲率

2. 積分核：
- 簡單平均：均勻權重
- 高斯平滑：權重隨距離衰減
- 用於降噪和平滑

3. d次導數的最小核大小：
- 最小大小 = d + 1
- 一階導數需要2個點
- 二階導數需要3個點
- 依此類推

注意事項：
1. 核大小影響計算精度
2. 較大的核提供更好的近似
3. 但計算成本也更高
4. 需要權衡精度和效率
